# 3D objektumdetekció CenterPointtal nuScenes adatokon

**Tárgy:** Alkalmazott AI a járműipari szoftverekben

Ebben a gyakorlatban egy teljes, reprodukálható MMDetection3D munkafolyamatot építünk fel Google Colabban: környezettelepítés, a nuScenes mini adatok előkészítése, scene-szintű train/val/test felosztás, előre tanított CenterPoint modell inferenciája és hivatalos nuScenes-kiértékelése.

## Tanulási célok

A notebook végére képes leszel:

- saját MMDetection3D forkot Colabban telepíteni;
- a nuScenes `v1.0-mini` részhalmazt letölteni és a `tools/create_data.py` segítségével feldolgozni;
- scene-szintű, módosítható arányú train/val/test splitet készíteni adatszivárgás nélkül;
- az `init_model` és `inference_detector` API-val pontfelhőn inferenciát futtatni;
- a CenterPoint modellt a nuScenes mAP, NDS és TP-hibametrikáival kiértékelni.

> **Colab:** válaszd a **Runtime / Change runtime type / T4 GPU** beállítást. A mini adatkészlet és a generált fájlok több GB tárhelyet igényelnek.

## 0. Környezet és telepítés

Az MMDetection3D v1.4.0 nem kompatibilis a Colab új Python 3.13 környezetével, ezért a notebook Python 3.10-es környezetet hoz létre. A GPU-t azonban nem a Python vagy a PyTorch telepíti: azt a Colabnak kell a virtuális géphez rendelnie.

Még az első kódcella előtt válaszd a **Runtime / Change runtime type / T4 GPU** beállítást. A cella először az `nvidia-smi` paranccsal ellenőrzi a fizikai GPU-t, és csak ezután cseréli le a Python-környezetet.

A telepítés során két egyszeri automatikus újraindítás történik:

1. a 3. cella Python 3.10-re vált;
2. az 5. cella telepíti a rögzített NumPy/PyTorch/MMCV bináris stacket, majd tiszta folyamatot indít.

Mindkét újracsatlakozás után futtasd ismét a notebookot a **Runtime / Run all** paranccsal. A markerfájlok miatt a kész lépések nem ismétlődnek. A stack PyTorch 2.1.0 + CUDA 12.1, NumPy 1.26.4 és MMCV 2.1.0 verziókat használ.

A projekt régi `setup.py develop` alapú editable telepítése ütközhet a modern pip-pel. Ezért a notebook egy `.pth` fájllal közvetlenül a klónozott fork forráskönyvtárát köti be a Python környezetbe. A fork módosításai így új csomagtelepítés nélkül érvényesülnek. Saját fork vagy kurzus-branch használatához módosítsd a `REPO_URL` és `REPO_REF` értékét.

In [ ]:
import platform
import subprocess
import sys
from pathlib import Path

REQUIRED_PYTHON = (3, 10)
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

def get_nvidia_smi():
    try:
        return subprocess.run(
            ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return ""

hardware_gpu = get_nvidia_smi()
assert hardware_gpu, (
    "A Colab nem rendelt GPU-t ehhez a runtime-hoz. Válaszd a "
    "Runtime / Change runtime type / T4 GPU beállítást, majd indítsd újra a cellát."
)
print(f"Colab GPU: {hardware_gpu}")

if sys.version_info[:2] != REQUIRED_PYTHON:
    assert IN_COLAB, "A notebook Python 3.10 környezetet igényel."
    print(
        f"A jelenlegi Python {platform.python_version()}; átváltás Python 3.10-re. "
        "A runtime egyszer automatikusan újraindul."
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "setuptools>=75",
            "wheel",
            "https://github.com/conda-incubator/condacolab/archive/main.zip",
        ],
        check=True,
    )
    import condacolab

    condacolab.install(python_version="3.10")
else:
    print(f"Kompatibilis Python-környezet: {platform.python_version()}")

In [ ]:
import os
import subprocess
from pathlib import Path

# Cseréld le a saját forkod URL-jére. Az alapérték önmagában is futtatható.
REPO_URL = "https://github.com/open-mmlab/mmdetection3d.git"
REPO_REF = "v1.4.0"
REPO_DIR = Path("/content/mmdetection3d")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f"Már létezik, nem klónozzuk újra: {REPO_DIR}")

os.chdir(REPO_DIR)
print(subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())

In [ ]:
import site
import subprocess
import sys
from pathlib import Path

from IPython import get_ipython

REPO_DIR = Path("/content/mmdetection3d")
STACK_MARKER = Path("/content/.mmdet3d_py310_torch210_cu121_data_v3_ready")
assert REPO_DIR.exists(), "Előbb futtasd a repository klónozását végző 4. cellát."

def pip_install(*packages, extra_args=(), quiet=True):
    command = [sys.executable, "-m", "pip", "install"]
    if quiet:
        command.append("-q")
    command.extend([*extra_args, *packages])
    print("Telepítés:", " ".join(packages))
    subprocess.run(command, check=True)

restart_required = not STACK_MARKER.exists()
if restart_required:
    # PyTorch 2.1 cpp_extension a pkg_resources.packaging exportot használja.
    pip_install("pip<25", "setuptools==69.5.1", "packaging==24.1", "wheel")
    pip_install(
        "torch==2.1.0+cu121",
        "torchvision==0.16.0+cu121",
        "torchaudio==2.1.0+cu121",
        extra_args=("--extra-index-url", "https://download.pytorch.org/whl/cu121"),
    )
    pip_install(
        "numpy==1.26.4",
        "mmengine==0.10.7",
        "mmdet==3.3.0",
        "nuscenes-devkit==1.2.0",
        "lyft-dataset-sdk==0.0.8",
        "scikit-image==0.24.0",
        "numba==0.60.0",
        "plyfile",
        "trimesh",
        "tensorboard",
    )
    pip_install(
        "mmcv==2.1.0",
        extra_args=("-f", "https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html"),
    )
else:
    print("A rögzített bináris és adatkonverter stack már telepítve van.")

source_path = str(REPO_DIR.resolve())
pth_path = Path(site.getsitepackages()[0]) / "mmdetection3d_source.pth"
pth_path.write_text(source_path + "\n", encoding="utf-8")
if source_path not in sys.path:
    sys.path.insert(0, source_path)
print(f"MMDetection3D forrás bekötve: {pth_path} -> {source_path}")

if restart_required:
    STACK_MARKER.touch()
    print("A bináris csomagok telepítve. A kernel most egyszer újraindul.")
    get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
import platform
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

REQUIRED_PYTHON = (3, 10)
REPO_DIR = Path("/content/mmdetection3d")
source_path = str(REPO_DIR.resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)

hardware_gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

import torch
import mmcv
import mmdet
import mmdet3d
import mmengine
from mmcv import ops
from pkg_resources import packaging as pkg_resources_packaging
from torch.utils.cpp_extension import CUDA_HOME

print(f"Python {platform.python_version()}")
print(f"PyTorch {torch.__version__}, CUDA runtime {torch.version.cuda}")
print(f"GPU hardver: {hardware_gpu}")
print(f"PyTorch GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nem elérhető'}")
print(f"MMEngine {mmengine.__version__}")
print(f"MMCV {mmcv.__version__}")
print(f"MMDetection {mmdet.__version__}")
print(f"MMDetection3D {mmdet3d.__version__}")
print(f"setuptools {version('setuptools')}, packaging {pkg_resources_packaging.__version__}")
print(f"CUDA_HOME: {CUDA_HOME}")

assert sys.version_info[:2] == REQUIRED_PYTHON
assert hardware_gpu, "A Colab runtime-hoz nincs GPU rendelve."
assert torch.version.cuda == "12.1"
assert torch.cuda.is_available(), (
    "Az nvidia-smi látja a GPU-t, de a PyTorch nem tudta inicializálni. "
    "Válaszd a Runtime / Disconnect and delete runtime parancsot, majd indíts "
    "új T4 GPU runtime-ot és futtasd újra a notebookot az elejétől."
)
assert callable(ops.nms)
print("A GPU, az MMCV CUDA-operátorok és a PyTorch build-eszközök elérhetők.")

## 1. A nuScenes mini letöltése

A nuScenes teljes trainval készlete több száz GB, ezért a gyakorlatban a 10 scene-ből és 404 annotált kulcsképkockából álló `v1.0-mini` változatot használjuk. Az archívum mérete megközelítőleg 4 GB.

A nuScenes egy **scene** alatt időben összefüggő vezetési szekvenciát ért. A későbbi felosztást scene-szinten kell elvégezni: ha ugyanazon útvonal szomszédos képkockái külön splitekbe kerülnének, az adatszivárgást okozna.

In [ ]:
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/mmdetection3d")
DATA_ROOT = REPO_DIR / "data" / "nuscenes"
ARCHIVE = Path("/content/v1.0-mini.tgz")
NUSCENES_URL = "https://www.nuscenes.org/data/v1.0-mini.tgz"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not ARCHIVE.exists():
    subprocess.run(["wget", "-c", NUSCENES_URL, "-O", str(ARCHIVE)], check=True)
else:
    print(f"Az archívum már létezik: {ARCHIVE}")

if not (DATA_ROOT / "v1.0-mini").exists():
    subprocess.run(["tar", "-xzf", str(ARCHIVE), "-C", str(DATA_ROOT)], check=True)
else:
    print("A v1.0-mini már ki van csomagolva.")

In [ ]:
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(version="v1.0-mini", dataroot=str(DATA_ROOT), verbose=False)
print(f"Scene-ek száma: {len(nusc.scene)}")
print(f"Annotált minták száma: {len(nusc.sample)}")
print(f"Első scene: {nusc.scene[0]['name']} ({nusc.scene[0]['description'][:80]}...)")

## 2. Info-fájlok készítése a `create_data.py` segítségével

A nyers nuScenes JSON táblák és szenzorfájlok helyett az MMDetection3D előre feldolgozott `.pkl` info-fájlokat használ. Ezek többek között a LiDAR-fájlok relatív útvonalát, a koordináta-transzformációkat és a 3D annotációkat tartalmazzák.

- `--version v1.0-mini`: a devkit rögzített `mini_train` és `mini_val` scene-listáit használja;
- `--max-sweeps 5`: egy kulcsképkockához legfeljebb öt korábbi LiDAR sweep metaadata kerül be;
- `--extra-tag nuscenes`: a kimeneti fájlnevek előtagja.

A parancs ground-truth adatbázist is készít az objektummintavételezéses augmentációhoz. Ez néhány percig tarthat.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/mmdetection3d")
DATA_ROOT = REPO_DIR / "data" / "nuscenes"
official_train_path = DATA_ROOT / "nuscenes_infos_train.pkl"
official_val_path = DATA_ROOT / "nuscenes_infos_val.pkl"
conversion_log = Path("/content/create_nuscenes_data.log")

if not (official_train_path.exists() and official_val_path.exists()):
    command = [
        sys.executable,
        "-m", "tools.create_data",
        "nuscenes",
        "--root-path", str(DATA_ROOT),
        "--out-dir", str(DATA_ROOT),
        "--extra-tag", "nuscenes",
        "--version", "v1.0-mini",
        "--max-sweeps", "5",
    ]
    process_env = os.environ.copy()
    process_env["PYTHONPATH"] = os.pathsep.join(
        [str(REPO_DIR), process_env.get("PYTHONPATH", "")]
    ).rstrip(os.pathsep)

    with conversion_log.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            env=process_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"A create_data.py hibával leállt (exit code: {return_code}). "
            f"A teljes napló itt található: {conversion_log}"
        )
else:
    print("Az info-fájlok már léteznek, az előkészítést kihagyjuk.")

In [ ]:
official_train = mmengine.load(official_train_path)
official_val = mmengine.load(official_val_path)

print(f"Hivatalos mini_train minták: {len(official_train['data_list'])}")
print(f"Hivatalos mini_val minták:   {len(official_val['data_list'])}")
print("Első rekord mezői:", sorted(official_train["data_list"][0].keys()))

assert len(official_train["data_list"]) + len(official_val["data_list"]) == len(nusc.sample)

## 3. Saját train/val/test split

A `create_data.py` a nuScenes devkit beépített mini felosztását követi, amely csak train és val részt definiál. Oktatási vagy fejlesztési célra az összes mini scene-t újraoszthatjuk három részre.

Az alábbi cellában a `SPLIT_RATIOS` értékei módosítják az arányokat; összegüknek 1-nek kell lennie. A `SEED` biztosítja a reprodukálhatóságot. Tíz scene mellett egy scene 10 százalékpontnak felel meg, ezért ennél finomabb arány itt nem értelmezhető.

> **Fontos:** saját splitnél a validációs metrikák összehasonlíthatósága megszűnik a hivatalos nuScenes benchmarkkal. A későbbi hivatalos NDS/mAP mérés ezért az érintetlen `nuscenes_infos_val.pkl` fájlt használja.

In [ ]:
import math
import random
from collections import defaultdict
from copy import deepcopy

SPLIT_RATIOS = {"train": 0.6, "val": 0.2, "test": 0.2}
SEED = 42

assert set(SPLIT_RATIOS) == {"train", "val", "test"}
assert math.isclose(sum(SPLIT_RATIOS.values()), 1.0)
assert all(ratio > 0 for ratio in SPLIT_RATIOS.values())

all_records = official_train["data_list"] + official_val["data_list"]
sample_to_scene = {sample["token"]: sample["scene_token"] for sample in nusc.sample}
scene_to_records = defaultdict(list)

for record in all_records:
    sample_token = record["token"]
    assert sample_token in sample_to_scene, f"Ismeretlen nuScenes sample token: {sample_token}"
    scene_to_records[sample_to_scene[sample_token]].append(record)

assert sum(map(len, scene_to_records.values())) == len(all_records)
scene_tokens = sorted(scene_to_records)
random.Random(SEED).shuffle(scene_tokens)
print(f"Felosztható scene-ek: {len(scene_tokens)}")

In [ ]:
def allocate_scene_counts(total, ratios):
    """Egész scene-darabszámok a legnagyobb maradék módszerével."""
    exact = {name: total * ratio for name, ratio in ratios.items()}
    counts = {name: int(value) for name, value in exact.items()}
    remainder = total - sum(counts.values())
    order = sorted(ratios, key=lambda name: exact[name] - counts[name], reverse=True)
    for name in order[:remainder]:
        counts[name] += 1
    return counts

scene_counts = allocate_scene_counts(len(scene_tokens), SPLIT_RATIOS)
assert all(count > 0 for count in scene_counts.values()), "Minden splithez legalább egy scene szükséges."

split_scenes = {}
start = 0
for split_name in ("train", "val", "test"):
    stop = start + scene_counts[split_name]
    split_scenes[split_name] = set(scene_tokens[start:stop])
    start = stop

assert set.union(*split_scenes.values()) == set(scene_tokens)
assert not (split_scenes["train"] & split_scenes["val"])
assert not (split_scenes["train"] & split_scenes["test"])
assert not (split_scenes["val"] & split_scenes["test"])

scene_name = {scene["token"]: scene["name"] for scene in nusc.scene}
for split_name, tokens in split_scenes.items():
    print(f"{split_name:>5}: {len(tokens)} scene - {sorted(scene_name[token] for token in tokens)}")

In [ ]:
SPLIT_WRITER_VERSION = "token-v2"
print(f"Custom split writer: {SPLIT_WRITER_VERSION}")

custom_paths = {}
custom_records = {}

for split_name, selected_scenes in split_scenes.items():
    records = [
        deepcopy(record)
        for scene_token in selected_scenes
        for record in scene_to_records[scene_token]
    ]
    for sample_idx, record in enumerate(records):
        record["sample_idx"] = sample_idx

    payload = {
        "metainfo": deepcopy(official_train["metainfo"]),
        "data_list": records,
    }
    output_path = DATA_ROOT / f"nuscenes_custom_infos_{split_name}.pkl"
    mmengine.dump(payload, output_path)
    custom_paths[split_name] = output_path
    custom_records[split_name] = records
    print(f"{split_name:>5}: {len(records):>3} minta -> {output_path.name}")

all_custom_tokens = [
    record["token"]
    for records in custom_records.values()
    for record in records
]
record_count = sum(map(len, custom_records.values()))
unique_token_count = len(set(all_custom_tokens))
assert record_count == len(all_records), (
    f"Mintavesztés történt: {record_count} / {len(all_records)} rekord maradt meg."
)
assert unique_token_count == len(all_records), (
    f"A tokenek nem egyediek: {unique_token_count} / {len(all_records)} egyedi token."
)
for split_name, records in custom_records.items():
    sample_indices = [record["sample_idx"] for record in records]
    assert sample_indices == list(range(len(records))), (
        f"A {split_name} split sample_idx mezői nem folytonosak."
    )
print("A splitek diszjunktak, minden token egyedi, a sample_idx mezők folytonosak.")

## 4. Előre tanított CenterPoint modell

A választott modell a 0,2 m-es pillar reprezentációt, SECOND backbone-t, SECFPN neck-et és circle NMS-t használó CenterPoint. A modellzoo adatai szerint körülbelül 4,6 GB GPU-memóriát igényel, ezért a nuScenes CenterPoint variánsok közül ez alkalmas leginkább ingyenes Colab GPU-ra.

A checkpointot a teljes nuScenes train spliten tanították. Az itt használt mini adatok célja az API és a kiértékelési folyamat gyakorlása, nem a közölt modellezoo-eredmény reprodukálása.

In [ ]:
MODEL_CONFIG = REPO_DIR / "configs/centerpoint/centerpoint_pillar02_second_secfpn_head-circlenms_8xb4-cyclic-20e_nus-3d.py"
CHECKPOINT = REPO_DIR / "checkpoints/centerpoint_pillar02_nuscenes.pth"
CHECKPOINT_URL = (
    "https://download.openmmlab.com/mmdetection3d/v1.0.0_models/centerpoint/"
    "centerpoint_02pillar_second_secfpn_circlenms_4x8_cyclic_20e_nus/"
    "centerpoint_02pillar_second_secfpn_circlenms_4x8_cyclic_20e_nus_20220811_031844-191a3822.pth"
)
CHECKPOINT.parent.mkdir(exist_ok=True)

if not CHECKPOINT.exists():
    subprocess.run(["wget", "-c", CHECKPOINT_URL, "-O", str(CHECKPOINT)], check=True)
else:
    print(f"A checkpoint már létezik: {CHECKPOINT}")

### Inference az MMDetection3D Python API-val

Az `init_model` a konfigurációból felépíti a hálózatot, betölti a checkpointot, GPU-ra mozgatja és evaluation módba állítja. Az `inference_detector` fájlútvonalat, NumPy-tömböt vagy ezek listáját fogadja.

A CenterPoint checkpoint 10 sweepből álló bemenetre készült. Egyetlen `.bin` fájl nem tartalmazza a korábbi sweep-ek útvonalát és transzformációját, ezért először a hivatalos `NuScenesDataset` tesztpipeline-jával állítjuk elő az időben és térben összeillesztett 5D ponttömböt. Ezt a tömböt adjuk át az `inference_detector` API-nak. Így az API-példa ugyanazt az inputreprezentációt használja, mint a teljes kiértékelés.

Az MMDetection3D egyes verziói közvetlenül egy `Det3DDataSample` objektumot adnak vissza, a kurzushoz használt fork viszont `(result, preprocessed_data)` tuple-t. A következő cella mindkét szerződést kezeli.

A predikált és ground-truth dobozok ugyanabban a **LiDAR-koordinátarendszerben** vannak: pozitív $x$ előre, pozitív $y$ balra, pozitív $z$ felfelé; a yaw a $z$ tengely körül értendő. A BEV-ábra közvetlenül az MMDetection3D által számított dobozsarkokat használja. A ground truth szaggatott, a predikció folytonos kontúrral jelenik meg; a szín mindkettőnél az objektumosztályt jelöli.

In [ ]:
from copy import deepcopy
from pathlib import Path

from mmengine.config import Config
from mmengine.registry import init_default_scope
from mmdet3d.apis import inference_detector, init_model
from mmdet3d.registry import DATASETS

REPO_DIR = Path("/content/mmdetection3d")
DATA_ROOT = REPO_DIR / "data" / "nuscenes"
official_val_path = DATA_ROOT / "nuscenes_infos_val.pkl"
assert official_val_path.exists(), f"Hiányzó annotációs fájl: {official_val_path}"

full_cfg = Config.fromfile(MODEL_CONFIG)
init_default_scope(full_cfg.get("default_scope", "mmdet3d"))

# Ugyanaz a 10-sweepes előfeldolgozás, amelyet a hivatalos tesztpipeline használ.
dataset_cfg = deepcopy(full_cfg.test_dataloader.dataset)
dataset_cfg["data_root"] = str(DATA_ROOT)
dataset_cfg["ann_file"] = official_val_path.name
dataset_cfg["lazy_init"] = False
test_dataset = DATASETS.build(dataset_cfg)
sample_index = 0
data_info = test_dataset.get_data_info(sample_index)
prepared_sample = test_dataset[sample_index]
multi_sweep_points = prepared_sample["inputs"]["points"].cpu().numpy()

# A ground truth ugyanabban a LiDAR-frame-ben van, mint a modell predikciói.
eval_ann_info = data_info["eval_ann_info"]
gt_labels = eval_ann_info["gt_labels_3d"]
gt_valid = gt_labels >= 0
gt_labels = gt_labels[gt_valid].astype(int)
gt_boxes_3d = eval_ann_info["gt_bboxes_3d"][gt_valid]
gt_box_corners = gt_boxes_3d.corners.detach().cpu().numpy()

# Az API már előfeldolgozott ponttömböt kap, ezért itt nem töltjük be újra a sweep-eket.
inference_cfg = Config.fromfile(MODEL_CONFIG)
inference_cfg.test_dataloader.dataset.pipeline = [
    dict(
        type="LoadPointsFromFile",
        coord_type="LIDAR",
        load_dim=5,
        use_dim=5,
    ),
    dict(type="Pack3DDetInputs", keys=["points"]),
]

model = init_model(inference_cfg, str(CHECKPOINT), device="cuda:0")
sample_token = data_info["token"]
sample = nusc.get("sample", sample_token)
lidar_record = nusc.get("sample_data", sample["data"]["LIDAR_TOP"])
lidar_path = DATA_ROOT / lidar_record["filename"]

inference_output = inference_detector(model, multi_sweep_points)
if isinstance(inference_output, tuple):
    result, inference_data = inference_output
else:
    result, inference_data = inference_output, None

assert hasattr(result, "pred_instances_3d"), type(result)
print(type(result).__name__, lidar_path)
print(f"API-bemenet: {len(multi_sweep_points):,} pont, {multi_sweep_points.shape[1]} jellemző")
print(f"Ground-truth objektumok: {len(gt_labels)}")
print("Időtartomány [s]:", multi_sweep_points[:, 4].min(), multi_sweep_points[:, 4].max())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

predictions = result.pred_instances_3d
score_threshold = 0.30
keep = predictions.scores_3d >= score_threshold
boxes = predictions.bboxes_3d.tensor[keep].detach().cpu().numpy()
box_corners = predictions.bboxes_3d[keep].corners.detach().cpu().numpy()
scores = predictions.scores_3d[keep].detach().cpu().numpy()
labels = predictions.labels_3d[keep].detach().cpu().numpy().astype(int)
classes = model.dataset_meta["classes"]
class_colors = {
    class_name: plt.get_cmap("tab10")(index % 10)
    for index, class_name in enumerate(classes)
}
columns = ["class", "score", "x", "y", "z", "length", "width", "height", "yaw"]

prediction_table = pd.DataFrame(
    [
        {
            "class": classes[label],
            "score": score,
            "x": box[0],
            "y": box[1],
            "z": box[2],
            "length": box[3],
            "width": box[4],
            "height": box[5],
            "yaw": box[6],
        }
        for box, score, label in zip(boxes, scores, labels)
    ],
    columns=columns,
)
print(f"{len(prediction_table)} detekció a {score_threshold:.2f} küszöb felett")
prediction_table.sort_values("score", ascending=False).head(20).style.format(
    {"score": "{:.3f}", "x": "{:.1f}", "y": "{:.1f}", "z": "{:.1f}", "yaw": "{:.2f}"}
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon

points = multi_sweep_points
keyframe_mask = np.isclose(points[:, 4], 0.0)
history_mask = ~keyframe_mask
fig, ax = plt.subplots(figsize=(11, 10))

ax.scatter(
    points[history_mask, 0],
    points[history_mask, 1],
    s=0.08,
    c="#c7ccd1",
    alpha=0.18,
    rasterized=True,
)
keyframe_scatter = ax.scatter(
    points[keyframe_mask, 0],
    points[keyframe_mask, 1],
    s=0.25,
    c=points[keyframe_mask, 2],
    cmap="viridis",
    vmin=-3,
    vmax=2,
    rasterized=True,
)

present_classes = set()

# Ground truth: osztályszín, vastag szaggatott kontúr.
for corners, label in zip(gt_box_corners, gt_labels):
    class_name = classes[label]
    color = class_colors[class_name]
    present_classes.add(class_name)
    footprint = corners[[0, 3, 7, 4], :2]
    ax.add_patch(
        Polygon(
            footprint,
            closed=True,
            fill=False,
            edgecolor=color,
            linewidth=2.6,
            linestyle="--",
            alpha=0.95,
            zorder=3,
        )
    )

# Predikció: osztályszín, folytonos kontúr és irányjelölő.
for corners, label, score in zip(box_corners, labels, scores):
    class_name = classes[label]
    color = class_colors[class_name]
    present_classes.add(class_name)
    footprint = corners[[0, 3, 7, 4], :2]
    ax.add_patch(
        Polygon(
            footprint,
            closed=True,
            fill=False,
            edgecolor=color,
            linewidth=1.8,
            linestyle="-",
            zorder=4,
        )
    )
    center = footprint.mean(axis=0)
    front_center = footprint[[2, 3]].mean(axis=0)
    ax.plot(
        [center[0], front_center[0]],
        [center[1], front_center[1]],
        color=color,
        linewidth=1.4,
        zorder=4,
    )

point_cloud_range = full_cfg.get("point_cloud_range", [-51.2, -51.2, -5, 51.2, 51.2, 3])
ax.set(
    xlim=(point_cloud_range[0], point_cloud_range[3]),
    ylim=(point_cloud_range[1], point_cloud_range[4]),
    aspect="equal",
    xlabel="x [m] - előre",
    ylabel="y [m] - balra",
    title=(
        f"CenterPoint, 10 sweep | GT: {len(gt_labels)}, "
        f"pred: {len(labels)} (score >= {score_threshold})"
    ),
)
ax.grid(alpha=0.15)
fig.colorbar(keyframe_scatter, ax=ax, fraction=0.035, pad=0.02, label="z [m]")

style_handles = [
    Line2D([0], [0], color="black", lw=2.6, linestyle="--", label="Ground truth"),
    Line2D([0], [0], color="black", lw=1.8, linestyle="-", label="Predikció"),
]
class_handles = [
    Line2D([0], [0], color=class_colors[name], lw=2, label=name)
    for name in classes
    if name in present_classes
]
ax.legend(
    handles=style_handles + class_handles,
    loc="upper right",
    fontsize=8,
    ncol=2,
)
plt.show()

## 5. Kiértékelés létező nuScenes metrikákkal

A `tools/test.py` a konfiguráció `NuScenesMetric` evaluatorát futtatja a hivatalos `mini_val` splittől származó 81 mintán. A legfontosabb metrikák:

- **mAP:** osztályonként és 0,5/1/2/4 méteres középponttávolság-küszöbökön számított átlagos pontosság átlaga;
- **mATE:** átlagos transzlációs hiba;
- **mASE:** átlagos skálahiba;
- **mAOE:** átlagos orientációs hiba;
- **mAVE:** átlagos sebességhiba;
- **mAAE:** átlagos attribútumhiba;
- **NDS:** a mAP és a normalizált TP-hibák összesített nuScenes Detection Score értéke.

$$\mathrm{NDS}=\frac{1}{10}\left(5\,\mathrm{mAP}+\sum_{mTP}\left[1-\min(1,mTP)\right]\right)$$

A mini eredmény statisztikailag zajos, és nem egyezik a modellzoo teljes validációs készleten közölt értékével. Modellválasztáskor azonos splitet és konfigurációt kell használni minden kísérlethez.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/mmdetection3d")
EVAL_WORK_DIR = REPO_DIR / "work_dirs/centerpoint_nuscenes_mini_eval"
RESULT_PREFIX = EVAL_WORK_DIR / "results"
EVAL_LOG = Path("/content/centerpoint_nuscenes_mini_eval.log")
EVAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable,
    "-m", "tools.test",
    str(MODEL_CONFIG),
    str(CHECKPOINT),
    "--work-dir", str(EVAL_WORK_DIR),
    "--cfg-options",
    f"test_evaluator.jsonfile_prefix={RESULT_PREFIX}",
    "test_dataloader.num_workers=2",
    "test_dataloader.persistent_workers=True",
]
process_env = os.environ.copy()
process_env["PYTHONPATH"] = os.pathsep.join(
    [str(REPO_DIR), process_env.get("PYTHONPATH", "")]
).rstrip(os.pathsep)

with EVAL_LOG.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=process_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"A tools.test hibával leállt (exit code: {return_code}). "
        f"A teljes napló itt található: {EVAL_LOG}"
    )

In [ ]:
metrics_path = RESULT_PREFIX / "pred_instances_3d" / "metrics_summary.json"
metrics = mmengine.load(metrics_path)
summary = pd.Series(
    {
        "mAP": metrics["mean_ap"],
        "NDS": metrics["nd_score"],
        "mATE": metrics["tp_errors"]["trans_err"],
        "mASE": metrics["tp_errors"]["scale_err"],
        "mAOE": metrics["tp_errors"]["orient_err"],
        "mAVE": metrics["tp_errors"]["vel_err"],
        "mAAE": metrics["tp_errors"]["attr_err"],
    },
    name="CenterPoint / nuScenes mini_val",
)
summary.to_frame().style.format("{:.4f}")

## 6. Rövid tanítás a saját spliten

A custom train info-fájl önmagában nem elég: a CenterPoint konfiguráció `ObjectSample` augmentációja külön ground-truth objektum-adatbázisból mintavételez. Ezt is kizárólag a custom train splitből kell újragenerálni, különben validációs vagy tesztobjektumok kerülhetnének a tanításba.

A következő cellák elkészítik az adatbázist és egy két epochos Colab-konfigurációt. A rövid futás csak a pipeline ellenőrzésére szolgál; érdemi modellhez több epoch, kísérletkövetés és gondos hiperparaméter-hangolás szükséges.

In [ ]:
from tools.dataset_converters.create_gt_database import create_groundtruth_database

CUSTOM_DB_INFO = DATA_ROOT / "nuscenes_custom_dbinfos_train.pkl"
if not CUSTOM_DB_INFO.exists():
    create_groundtruth_database(
        "NuScenesDataset",
        str(DATA_ROOT),
        "nuscenes_custom",
        info_path=custom_paths["train"].name,
    )
else:
    print(f"A custom ground-truth adatbázis már létezik: {CUSTOM_DB_INFO}")

In [ ]:
TRAIN_CONFIG = REPO_DIR / "configs/centerpoint/centerpoint_pillar02_nuscenes_mini_custom.py"
training_cfg = Config.fromfile(MODEL_CONFIG)
training_cfg.train_dataloader.batch_size = 2
training_cfg.train_dataloader.num_workers = 2
training_cfg.train_dataloader.persistent_workers = True

train_dataset_cfg = training_cfg.train_dataloader.dataset
while "dataset" in train_dataset_cfg:
    train_dataset_cfg = train_dataset_cfg["dataset"]
train_dataset_cfg.ann_file = custom_paths["train"].name

object_sampler = next(
    transform
    for transform in train_dataset_cfg.pipeline
    if transform["type"] == "ObjectSample"
)
object_sampler["db_sampler"]["info_path"] = str(CUSTOM_DB_INFO)
training_cfg.train_cfg.max_epochs = 2
training_cfg.train_cfg.val_interval = 1
training_cfg.default_hooks.checkpoint.interval = 1
training_cfg.optim_wrapper.optimizer.lr = 1e-4
training_cfg.dump(TRAIN_CONFIG)
print(TRAIN_CONFIG)

In [ ]:
RUN_SHORT_TRAINING = False
TRAIN_WORK_DIR = REPO_DIR / "work_dirs/centerpoint_nuscenes_mini_custom"

if RUN_SHORT_TRAINING:
    subprocess.run(
        [
            sys.executable,
            "tools/train.py",
            str(TRAIN_CONFIG),
            "--work-dir", str(TRAIN_WORK_DIR),
            "--amp",
        ],
        cwd=REPO_DIR,
        check=True,
    )
    print(f"Checkpoint: {TRAIN_WORK_DIR / 'epoch_2.pth'}")
else:
    print("A tanítás kihagyva. Állítsd a RUN_SHORT_TRAINING értékét True-ra a futtatáshoz.")

## 7. Kimeneti szerződés a későbbi ROS2-integrációhoz

Az inference eredménye közvetlenül PyTorch/MMDetection3D objektumokat tartalmaz. ROS2 node-ban ezeket CPU-n lévő egyszerű számokká kell alakítani, majd például `vision_msgs/Detection3DArray` üzenetre leképezni.

Különösen figyelj a következőkre:

- az üzenet `header.stamp` értéke az eredeti szenzoridő legyen;
- a `frame_id` és a dobozok koordinátarendszere egyezzen;
- a nuScenes LiDAR doboz formátuma `(x, y, z, dx, dy, dz, yaw, vx, vy)`;
- a score-küszöb és az osztályazonosító-leképezés legyen konfigurálható, ne legyen a callbackbe égetve.

In [ ]:
ros_ready_predictions = [
    {
        "class_id": int(label),
        "class_name": classes[label],
        "score": float(score),
        "center_xyz": box[:3].tolist(),
        "size_xyz": box[3:6].tolist(),
        "yaw": float(box[6]),
        "velocity_xy": box[7:9].tolist() if len(box) >= 9 else [0.0, 0.0],
    }
    for box, score, label in zip(boxes, scores, labels)
]

message_payload = {
    "frame_id": "LIDAR_TOP",
    "timestamp_us": sample["timestamp"],
    "detections": ros_ready_predictions,
}
message_payload["detections"][:2]

## Összefoglalás

- Az MMDetection3D fork `.pth` alapú forrásbekötése új csomagépítés nélkül teszi elérhetővé a hallgatói módosításokat.
- A `create_data.py` a nyers nuScenes struktúrából MMDetection3D v2 info-fájlokat és ground-truth adatbázist készít.
- Idősoros szenzoradatot scene-szinten kell felosztani, különben közeli képkockákon keresztül adatszivárgás keletkezik.
- Az `init_model` betölti a konfigurációt és a checkpointot; az `inference_detector` egy vagy több pontfelhőn futtat predikciót.
- A hivatalos nuScenes értékelés mAP, NDS, mATE, mASE, mAOE, mAVE és mAAE értékeket közöl, és a devkit rögzített validációs splitjét várja.
- Saját split esetén az info-fájl mellett az augmentációs ground-truth adatbázist is újra kell készíteni.

## Önálló feladatok

1. A 14. cellában módosítsd a splitet 70/20/10 arányúra és a seedet `7`-re. Hány scene és minta kerül az egyes részekbe?
2. A 21. cellában vizsgáld meg, hogyan változik a detekciók száma 0,1; 0,3 és 0,5 score-küszöb mellett.
3. A 25. cella eredményéből keresd meg a legjobb és legrosszabb osztályt az AP alapján. Adj adatalapú magyarázatot a különbségre.
4. A 28. cellában készített konfigurációban állíts be 4 epochot, majd értékeld a kapott checkpointot ugyanazzal a hivatalos mini-validációval.
5. Tervezd meg, mely `vision_msgs/Detection3DArray` mezőkbe kerülnének a 31. cellában előállított adatok, és hol végeznéd el a LiDAR-frame és a jármű-frame közötti transzformációt.

<!--
Megoldási támpontok:

1. Tíz scene miatt a legnagyobb maradék módszer egész scene-ekre kerekít; mindig ellenőrizd a cella által kiírt minta-darabszámokat is.
2. A detekciók száma a küszöb növelésével monoton nem nőhet. Vesd össze, mely osztályok tűnnek el először.
3. Használd a metrics["label_aps"] szótár osztályonkénti távolságküszöbeit, és vesd össze az osztályok mini-validációs előfordulásával.
4. Állítsd training_cfg.train_cfg.max_epochs = 4 értékre, majd a tesztparancs checkpoint argumentumaként add meg az epoch_4.pth fájlt. A split és minden más beállítás maradjon azonos.
5. A középpont és méret a bounding box pose/size mezőibe, a score és class ID a hypothesis mezőibe kerül. A koordináta-transzformációt időbélyeggel helyesen lekérdezett TF2 transzformációval, publikálás előtt végezd el.
-->